# Fine-tune Transkun V2
This notebook prepares and launches a true fine-tuning run from V2 pretrained weights, then provides optional validation and evaluation steps.

Why this workflow: in this codebase, if a checkpoint already exists, the trainer restores optimizer and LR scheduler state from that checkpoint. So we seed a fresh checkpoint with pretrained model weights but a new low-LR schedule for fine-tuning.

In [ ]:
from pathlib import Path
import copy
import pickle
import shutil
import shlex
import subprocess
import sys

import torch

PROJECT_ROOT = Path("/scratch/gilbreth/li5042/transkun/transkun_fork")
FINE_TUNE_ROOT = PROJECT_ROOT / "eval_utils/4_fine_tuning"
TRAINING_DATA_DIR = FINE_TUNE_ROOT / "model_info/training_data"
MODEL_PARAMS_DIR = FINE_TUNE_ROOT / "model_info/model_params"
RUN_DIR = FINE_TUNE_ROOT / "output/finetune_v2_run"
RUN_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = Path("/scratch/gilbreth/li5042/datasets/MAESTRO")
TRAIN_META = TRAINING_DATA_DIR / "train.pickle"
VAL_META = TRAINING_DATA_DIR / "val.pickle"
TEST_META = TRAINING_DATA_DIR / "test.pickle"

PRETRAINED_CKPT = MODEL_PARAMS_DIR / "checkpoint.pt"
PRETRAINED_CONF = MODEL_PARAMS_DIR / "model.conf"

FINETUNE_CKPT = RUN_DIR / "checkpoint_finetuned.pt"
FINETUNE_CONF = RUN_DIR / "model_finetune.conf"

N_PROCESS = 1
BATCH_SIZE = 4
DATA_LOADER_WORKERS = 8
MAX_LR = 1e-4
WEIGHT_DECAY = 1e-4
N_ITER = 50000

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")
print(f"Run directory: {RUN_DIR}")

## 1) Validate Inputs

In [ ]:
required_paths = [
    DATASET_PATH,
    TRAIN_META,
    VAL_META,
    TEST_META,
    PRETRAINED_CKPT,
    PRETRAINED_CONF,
]

missing = [p for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required inputs:\n" + "\n".join(str(p) for p in missing))

print("All required paths exist.")
for p in required_paths:
    print(f"  - {p}")

## 2) Seed a Fine-Tuning Checkpoint (Pretrained Weights + Fresh Optimizer/LR Schedule)

In [ ]:
import moduleconf
from transkun.TrainUtil import initializeCheckpoint, save_checkpoint

# Keep a run-local copy of the model config used for this fine-tune run.
shutil.copy2(PRETRAINED_CONF, FINETUNE_CONF)

conf_manager = moduleconf.parseFromFile(str(FINETUNE_CONF))
TransKun = conf_manager["Model"].module.TransKun
conf = conf_manager["Model"].config

# Create fresh optimizer/scheduler state using fine-tune hyperparameters.
start_epoch, start_iter, model, loss_tracker, best_state_dict, optimizer, lr_scheduler = initializeCheckpoint(
    TransKun,
    device=DEVICE,
    max_lr=MAX_LR,
    weight_decay=WEIGHT_DECAY,
    nIter=N_ITER,
    conf=conf,
 )

# Load pretrained model weights into the fresh training state.
pretrained_ckpt = torch.load(PRETRAINED_CKPT, map_location=DEVICE)
source_state = pretrained_ckpt.get("best_state_dict", pretrained_ckpt.get("state_dict", pretrained_ckpt))
load_result = model.load_state_dict(source_state, strict=False)

best_state_dict = copy.deepcopy(model.state_dict())
save_checkpoint(
    str(FINETUNE_CKPT),
    epoch=0,
    nIter=0,
    model=model,
    lossTracker=loss_tracker,
    best_state_dict=best_state_dict,
    optimizer=optimizer,
    lrScheduler=lr_scheduler,
 )

print(f"Seeded fine-tune checkpoint: {FINETUNE_CKPT}")
print(f"Run config file: {FINETUNE_CONF}")
print(f"Missing keys: {len(load_result.missing_keys)}")
print(f"Unexpected keys: {len(load_result.unexpected_keys)}")

## 3) Launch Fine-Tuning

In [ ]:
train_cmd = [
    sys.executable,
    "-m",
    "transkun.train",
    "--nProcess",
    str(N_PROCESS),
    "--datasetPath",
    str(DATASET_PATH),
    "--datasetMetaFile_train",
    str(TRAIN_META),
    "--datasetMetaFile_val",
    str(VAL_META),
    "--batchSize",
    str(BATCH_SIZE),
    "--dataLoaderWorkers",
    str(DATA_LOADER_WORKERS),
    "--max_lr",
    str(MAX_LR),
    "--weight_decay",
    str(WEIGHT_DECAY),
    "--nIter",
    str(N_ITER),
    "--modelConf",
    str(FINETUNE_CONF),
    str(FINETUNE_CKPT),
]

print("Training command:")
print(shlex.join(train_cmd))

RUN_TRAIN = False
if RUN_TRAIN:
    subprocess.run(train_cmd, cwd=str(PROJECT_ROOT), check=True)
else:
    print("Set RUN_TRAIN = True to start fine-tuning.")
    print("Checkpoint and scheduler state come from the seeded checkpoint file above.")

## 4) Optional: Quick Validation on One Test Audio

In [ ]:
with open(TEST_META, "rb") as f:
    test_entries = pickle.load(f)

if not test_entries:
    raise RuntimeError("test.pickle is empty")

sample_test_wav = DATASET_PATH / test_entries[0]["audio_filename"]
validation_mid = RUN_DIR / "validation_test.mid"

transcribe_cmd = [
    sys.executable,
    "-m",
    "transkun.transcribe",
    str(sample_test_wav),
    str(validation_mid),
    "--device",
    DEVICE,
    "--weight",
    str(FINETUNE_CKPT),
    "--conf",
    str(FINETUNE_CONF),
]

print("Validation command:")
print(shlex.join(transcribe_cmd))

RUN_VALIDATION = False
if RUN_VALIDATION:
    subprocess.run(transcribe_cmd, cwd=str(PROJECT_ROOT), check=True)
else:
    print("Set RUN_VALIDATION = True to run one-file validation transcription.")

## 5) Optional: Transcribe Test Split Only and Compute Metrics

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import queue

PRED_DIR = RUN_DIR / "predicted_midis_test"
PRED_DIR.mkdir(parents=True, exist_ok=True)

with open(TEST_META, "rb") as f:
    test_entries = pickle.load(f)

test_wavs = [DATASET_PATH / e["audio_filename"] for e in test_entries]
print(f"Test files listed in metadata: {len(test_wavs)}")

available_gpus = torch.cuda.device_count()
print(f"Detected GPUs: {available_gpus}")

RUN_TEST_TRANSCRIPTION = False

if RUN_TEST_TRANSCRIPTION:
    if available_gpus < 1:
        raise RuntimeError("No CUDA GPU available for multi-GPU transcription")

    gpu_queue = queue.Queue()
    for gpu_id in range(available_gpus):
        gpu_queue.put(gpu_id)

    def transcribe_one(wav_path):
        gpu_id = gpu_queue.get()
        try:
            out_midi = PRED_DIR / f"{wav_path.stem}_pred.mid"
            cmd = [
                sys.executable,
                "-m",
                "transkun.transcribe",
                str(wav_path),
                str(out_midi),
                "--device",
                f"cuda:{gpu_id}",
                "--weight",
                str(FINETUNE_CKPT),
                "--conf",
                str(FINETUNE_CONF),
            ]
            proc = subprocess.run(cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True)
            return wav_path, proc.returncode, proc.stderr
        finally:
            gpu_queue.put(gpu_id)

    failures = []
    with ThreadPoolExecutor(max_workers=available_gpus) as pool:
        futures = [pool.submit(transcribe_one, wav_path) for wav_path in test_wavs]
        for future in as_completed(futures):
            wav_path, rc, err = future.result()
            if rc != 0:
                failures.append((wav_path, rc, err[-2000:]))

    print(f"Transcription complete. Failed files: {len(failures)}")
    if failures:
        print("First failure sample:")
        print(failures[0][0])
        print(failures[0][2])
else:
    print("Set RUN_TEST_TRANSCRIPTION = True to transcribe the test split only.")

In [ ]:
metrics_cmd = [
    sys.executable,
    str(PROJECT_ROOT / "eval_utils/2_recreate_metrics/compute_comparison_metrics.py"),
    "--maestro_dir",
    str(DATASET_PATH),
    "--est_dir",
    str(PRED_DIR),
    "--workers",
    "16",
]

print("Metrics command:")
print(shlex.join(metrics_cmd))

RUN_METRICS = False
if RUN_METRICS:
    subprocess.run(metrics_cmd, cwd=str(PROJECT_ROOT), check=True)
else:
    print("Set RUN_METRICS = True after test predictions exist in PRED_DIR.")